# VAR 2026 Colab runner
Run every cell top to bottom. Safe to re-run after a disconnect: cell 1 skips re-cloning if the repo is already present, cell 3's `real_train_fn` skips scenes that already finished training, and all output lives on Drive.

In [ ]:
import os

REPO_URL = "https://github.com/viethoang2503/viettel-ai-race-bts-digital.git"
REPO_DIR = "/content/viettel-ai-race-bts-digital"

if not os.path.isdir(REPO_DIR):
    !git clone --recurse-submodules {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
!bash environment/setup_colab.sh

In [ ]:
import functools
from pathlib import Path

from src.common.config import load_scenes
from src.evaluation.compute_metrics import load_lpips_model
from src.orchestrator.run_pipeline import run_baseline_pipeline
from src.rendering.gs_render_fn import real_render_fn
from src.training.gs_train_fn import real_train_fn

ITERATIONS = 30000
OUTPUT_ROOT = Path("/content/drive/MyDrive/var2026/outputs")

scenes = load_scenes()
train_fn = functools.partial(real_train_fn, iterations=ITERATIONS)

result = run_baseline_pipeline(
    scenes=scenes,
    train_fn=train_fn,
    render_fn=real_render_fn,
    lpips_model=load_lpips_model(),
    psnr_max=30.0,
    output_root=OUTPUT_ROOT,
)

In [ ]:
print("Per-scene scores:")
for name, score in result.per_scene_scores.items():
    print(f"  {name}: {score:.4f}")

if result.skipped_scenes:
    print("\nSkipped scenes:")
    for name, problems in result.skipped_scenes.items():
        print(f"  {name}: {problems}")

if result.validation_problems:
    print("\nValidation problems (submission withheld):")
    for problem in result.validation_problems:
        print(f"  {problem}")

print(f"\nsubmission_zip: {result.submission_zip}")